# Level 1: fit and save the final development-trained model

Use the development-leading larger architecture `50 -> 64 -> 32 -> 2`, AdamW (`lr=0.001`, `weight_decay=0.0001`), batch size 64, model seed 42, and shuffle seed 42. Train a fresh model on **all five development folds: 54 simulations, 810 maps**.

**Fixed duration: 11 epochs.** This is the median of the larger model's best validation epochs `[3, 6, 18, 11, 19]`. It is a simple development-based rule, not a proven optimum for the larger training set. There is no validation checkpoint selection in final fitting.

Fit new feature and target scalers on the full development training set. Save weights, scalers, settings, and loss history together. This notebook never loads or evaluates test data.

Select `.venv/bin/python`, restart the kernel after module edits, and run top to bottom. A saved checkpoint is reloaded on reruns; delete nothing automatically. To intentionally retrain and save another model, choose a new checkpoint filename.


## 1. Locate the project and import final-training helpers

`final_train.py` separates full-development fitting from `experiment.py`, which holds out a validation fold.

In [1]:
from pathlib import Path
import sys
import json

PROJECT_ROOT = next(
    (p for p in (Path.cwd(), *Path.cwd().parents)
     if (p / "mlp" / "data.py").is_file()), None,
)
if PROJECT_ROOT is None:
    raise RuntimeError("Open the notebook inside the cosmoreg project.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import torch
from mlp import data
from mlp.final_train import train_final_model, save_final_model, load_final_model

LEVEL_ROOT = PROJECT_ROOT / "mlp" / "level1"
SPECTRA_ROOT = PROJECT_ROOT / "powerspectrum"
CHECKPOINT_PATH = LEVEL_ROOT / "checkpoints" / "level1_larger_seed42_epochs11.pt"
print("Project:", PROJECT_ROOT)


Project: /home/meatpigeon/Personal/classes/csci494/cosmoreg


## 2. Fix the settings before test evaluation

The choices below match the completed configuration comparison. Read its summary to verify the larger model led the predefined ranking and that its median best epoch is 11. Do not select a duration by looking at test scores.

In [2]:
FINAL_SETTINGS = {
    "hidden_dims": (64, 32),
    "epochs": 11,
    "learning_rate": 1e-3,
    "weight_decay": 1e-4,
    "batch_size": 64,
    "seed": 42,
    "shuffle_seed": 42,
}
comparison = json.loads((LEVEL_ROOT / "level1_config_comparison.json").read_text())
assert comparison["ranking"][0] == "larger"
best_epochs = comparison["summaries"]["larger"]["best_epochs"]
assert int(np.median(best_epochs)) == FINAL_SETTINGS["epochs"]
for key, value in comparison["configs"]["larger"].items():
    if key != "epochs":
        assert list(value) == list(FINAL_SETTINGS[key]) if key == "hidden_dims" else value == FINAL_SETTINGS[key]
print("Chosen larger architecture:", FINAL_SETTINGS["hidden_dims"])
print("Development best epochs:", best_epochs)
print("Fixed final duration:", FINAL_SETTINGS["epochs"])


Chosen larger architecture: (64, 32)
Development best epochs: [3, 6, 18, 11, 19]
Fixed final duration: 11


## 3. Train and save, or reload the existing checkpoint

All development rows are joined; maps/spectra are not averaged during training. Each sample keeps its original target. Both scalers are newly fitted on all 810 development maps.

The epoch losses below combine batch losses while weights are changing. They are training measurements, not validation or test scores. The final epoch's weights are kept because the duration was fixed beforehand.

Small CPU matrices use one thread during fitting; the previous thread count is restored afterward. Checkpoints are local generated artifacts excluded by `.gitignore`.

In [3]:
if CHECKPOINT_PATH.exists():
    final_results = load_final_model(CHECKPOINT_PATH)
    print("Reloaded existing checkpoint:", CHECKPOINT_PATH)
else:
    previous_threads = torch.get_num_threads()
    torch.set_num_threads(1)
    try:
        final_results = train_final_model(SPECTRA_ROOT, level=1, **FINAL_SETTINGS)
    finally:
        torch.set_num_threads(previous_threads)
    final_results["settings"]["configuration_name"] = "larger"
    final_results["settings"]["epoch_selection"] = "median best epoch from five development folds"
    final_results["settings"]["development_best_epochs"] = best_epochs
    save_final_model(final_results, CHECKPOINT_PATH)
    print("Saved checkpoint:", CHECKPOINT_PATH)

assert final_results["settings"]["level"] == 1
for key, value in FINAL_SETTINGS.items():
    saved_value = final_results["settings"][key]
    assert list(saved_value) == list(value) if key == "hidden_dims" else saved_value == value
assert final_results["training_maps"] == 810
assert len(final_results["training_simulation_ids"]) == 54
assert len(final_results["train_history"]) == FINAL_SETTINGS["epochs"]
print("Training maps:", final_results["training_maps"])
print("Training simulations:", len(final_results["training_simulation_ids"]))
print("Checkpoint size (KiB):", round(CHECKPOINT_PATH.stat().st_size / 1024, 1))


Epoch 01/11 | Training MSE: 0.8984
Epoch 02/11 | Training MSE: 0.7936
Epoch 03/11 | Training MSE: 0.7789
Epoch 04/11 | Training MSE: 0.7617
Epoch 05/11 | Training MSE: 0.7509
Epoch 06/11 | Training MSE: 0.7501
Epoch 07/11 | Training MSE: 0.7410
Epoch 08/11 | Training MSE: 0.7342
Epoch 09/11 | Training MSE: 0.7280
Epoch 10/11 | Training MSE: 0.7192
Epoch 11/11 | Training MSE: 0.7167
Saved checkpoint: /home/meatpigeon/Personal/classes/csci494/cosmoreg/mlp/level1/checkpoints/level1_larger_seed42_epochs11.pt
Training maps: 810
Training simulations: 54
Checkpoint size (KiB): 26.5


## 4. Verify saved preprocessing and prediction roundtrip

Reload the checkpoint and compare its predictions with the in-memory model using development inputs. This verifies serialization; it is not an independent performance evaluation.

Check that saved scalers match statistics fitted on all development rows. Convert predictions back to original units to confirm shape and finite values. No test data is needed for these checks.

In [4]:
loaded_results = load_final_model(CHECKPOINT_PATH)
X_dev, y_dev, ids_dev = data.load_development(SPECTRA_ROOT, 1)
s = loaded_results["scalers"]
feature_mean, feature_std = data.fit_feature_scaler(X_dev)
target_mean, target_std = data.fit_target_scaler(y_dev)
for name, expected in (
    ("feature_mean", feature_mean), ("feature_std", feature_std),
    ("target_mean", target_mean), ("target_std", target_std),
):
    assert np.array_equal(s[name], expected)
assert loaded_results["training_simulation_ids"] == np.unique(ids_dev).tolist()
X_tensor = torch.from_numpy(data.transform_features(X_dev, s["feature_mean"], s["feature_std"]))
with torch.no_grad():
    before_save = final_results["model"](X_tensor)
    after_load = loaded_results["model"](X_tensor)
assert torch.equal(before_save, after_load)
predictions_original = data.inverse_transform_targets(after_load.numpy(), s["target_mean"], s["target_std"])
assert predictions_original.shape == (810, 2)
assert np.isfinite(predictions_original).all()
assert loaded_results["target_names"] == ["Omega_m", "sigma_8"]
print("PASS: full-development scaler statistics and saved-model predictions verified.")
print("Test data remains untouched. The next step is evaluating this fixed checkpoint.")


PASS: full-development scaler statistics and saved-model predictions verified.
Test data remains untouched. The next step is evaluating this fixed checkpoint.
